In [0]:
%run "./School-Setup"

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_school}/enrollments-json-raw")
display(files)

In [0]:
%python
import pyspark.sql.functions as F

(spark.readStream
           .format("cloudFiles")
           .option("cloudFiles.format", "json")
           .option("cloudFiles.inferColumnTypes","true")
           .option("cloudFiles.schemaLocation", f"{checkpoint_path}/enrollments_bronze")
           .load(f"{dataset_school}/enrollments-json-raw")
           .select("*",
                   F.current_timestamp().alias("arrival_time"),
                   F.col("_metadata.file_path").alias("source_file"))
     .writeStream
           .format("delta")
           .option("checkpointLocation", f"{checkpoint_path}/enrollments_bronze")
           .outputMode("append")
           .trigger(availableNow=True)
           .table("enrollments_bronze")
)


In [0]:
%sql
SELECT * FROM enrollments_bronze

In [0]:
%sql
SELECT count(1) FROM enrollments_bronze

In [0]:
%python
load_new_data()

In [0]:
%python
students_lookup_df = (spark.read
                       .format("json")
                       .load(f"{dataset_school}/students-json"))

In [0]:
%python
display(students_lookup_df)

In [0]:
%python
enrollments_enriched_df = (spark.readStream
     .table("enrollments_bronze")
     .where("quantity > 0")
     .withColumn("formatted_timestamp", F.from_unixtime("enroll_timestamp", "yyyy-MM-dd HH:mm:ss").cast("timestamp") )
     .join(students_lookup_df, "student_id")
     .select("enroll_id", "quantity", "student_id", "email", "formatted_timestamp", "courses")
)

In [0]:
%python
(enrollments_enriched_df.writeStream
                       .format("delta")
                       .option("checkpointLocation", f"{checkpoint_path}/enrollments_silver")
                       .outputMode("append")
                       .trigger(availableNow=True)
                       .table("enrollments_silver"))

In [0]:
%sql
SELECT * FROM enrollments_silver

In [0]:
%python
enrollments_agg_df = (spark.readStream
                            .table("enrollments_silver")
                            .withColumn("day", F.date_trunc("DD", "formatted_timestamp"))
                            .groupBy("student_id", "email", "day")
                            .agg(F.sum("quantity").alias("courses_counts"))
                            .select("student_id", "email", "day", "courses_counts")
)

In [0]:
%python
(enrollments_agg_df.writeStream
                   .format("delta")
                   .outputMode("complete")
                   .option("checkpointLocation", f"{checkpoint_path}/daily_student_courses")
                   .trigger(availableNow=True)
                   .table("daily_student_courses"))

In [0]:
%sql
SELECT * FROM daily_student_courses

In [0]:
%python
for s in spark.streams.active:
   print("Stopping stream: " + s.id)
   s.stop()
   s.awaitTermination()